# 🚗 Automatic Number Plate Recognition (ANPR) — YOLOv8 Training
### Project: Computer Vision Based Road Safety Monitoring System
**Author:** Shibchandan Mistry  
**Module:** ANPR (License Plate Detection & Text Recognition)

---
> **Tip:** Before running, ensure you have enabled GPU in Colab:  
> Click **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU** ➔ Click **Save**.

## Step 1: Verify GPU Acceleration

In [ ]:
!nvidia-smi

## Step 2: Install Ultralytics (YOLOv8) & Roboflow

In [ ]:
!pip install -q ultralytics roboflow

## Step 3: Download License Plate Dataset
We use the **Roboflow Universe License Plate Recognition Dataset** (YOLOv8 format).

If you have your own Roboflow API key (free at [roboflow.com](https://roboflow.com)), enter it below.  
Otherwise, the cell downloads a public benchmark license plate dataset automatically.

In [ ]:
import os

# Method A: Download using Roboflow API (replace with your key if you have one)
ROBOFLOW_API_KEY = ""  # Optional: Enter your key here

if ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
    version = project.version(1)
    dataset = version.download("yolov8")
    data_yaml = f"{dataset.location}/data.yaml"
else:
    # Method B: Direct public download of YOLOv8 License Plate Dataset
    !curl -L "https://universe.roboflow.com/ds/x0R6c3v1U9?key=W4c1n8d4vA" > roboflow.zip; unzip -q roboflow.zip -d ./dataset; rm roboflow.zip
    data_yaml = "/content/dataset/data.yaml"

print(f"\nDataset ready at: {data_yaml}")

## Step 4: Train YOLOv8 Nano Model
- **Base model:** `yolov8n.pt` (Pretrained Nano model - fast & lightweight)
- **Epochs:** `30` (takes ~5–8 minutes on free T4 GPU)
- **Image size:** `640`
- **Batch size:** `16`

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 Nano
model = YOLO("yolov8n.pt")

# Start Training
results = model.train(
    data=data_yaml,
    epochs=30,
    imgsz=640,
    batch=16,
    project="anpr_training",
    name="yolov8n_plate",
    device=0
)

print("\nTraining complete!")

## Step 5: View Training Metrics & Accuracy Graphs
These graphs (mAP50, Precision, Recall, Loss) are perfect to include in your **Project Report & Presentation**.

In [ ]:
from IPython.display import Image, display
import os

results_img = "anpr_training/yolov8n_plate/results.png"
conf_matrix = "anpr_training/yolov8n_plate/confusion_matrix.png"

if os.path.exists(results_img):
    print("--- Training & Validation Metrics (Loss, Precision, Recall, mAP) ---")
    display(Image(results_img, width=800))

if os.path.exists(conf_matrix):
    print("--- Confusion Matrix ---")
    display(Image(conf_matrix, width=600))

## Step 6: Test Detection on Validation Images

In [ ]:
# Load the newly trained best weights
trained_model = YOLO("anpr_training/yolov8n_plate/weights/best.pt")

# Run validation
metrics = trained_model.val(data=data_yaml)

print(f"\nEvaluation Results:")
print(f"  Precision : {metrics.box.mp:.4f}")
print(f"  Recall    : {metrics.box.mr:.4f}")
print(f"  mAP@0.50  : {metrics.box.map50:.4f}")
print(f"  mAP@0.50:0.95 : {metrics.box.map:.4f}")

## Step 7: Download `best.pt` to Use in Your Project
Running this cell will automatically download your trained `best.pt` weight file to your local computer.
Place this downloaded file at: `results/anpr/weights/best.pt` in your project folder!

In [ ]:
from google.colab import files

weights_path = "anpr_training/yolov8n_plate/weights/best.pt"
print(f"Downloading {weights_path} to your computer...")
files.download(weights_path)